## Author: Nathaniel Ribeiro
## Date: 8/20/26

In [1]:
import lifelines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
DATA_DIR = Path("/data") / "irb" / "pharmacometrics" / "pro00117942"
METADATA_PATH = DATA_DIR / "Duke_CoMPAS_07142026_deidentified_v3.csv"
df = pd.read_csv(METADATA_PATH)

In [3]:
df

,record_id,redcap_data_access_group,study_id,sex_v2,age_calculated,site,ileal_slide,colon_slide,subject_race___1,subject_race___2,...,age_at_induction_date,age_at_dose_date,age_at_fu_date,age_at_change_intherapy_date,age_at_fu_height_date,age_at_fu_date_of_blood_draw,age_at_fu_cmp_date,age_at_fu_calprotectin_date,age_at_fu_colonoscopy_date,age_at_yr_fu_date
0,1,NaN,DU-R-0004,NaN,NaN,NaN,NaN,NaN,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,NaN,DU-R-0176,1.0,15.40,3.0,SP20-006943,SP20-006943,1,0,...,15.630390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DU-R-0016,NaN,DU-R-0016,2.0,10.19,NaN,SX12-041669,SX12-041669,0,1,...,12.246407,NaN,11.200548,NaN,11.200548,11.164956,11.164956,NaN,NaN,14.956879
3,DU-R-0017,NaN,DU-R-0017,2.0,14.97,NaN,SP15-033985,SP15-033985,0,1,...,15.964408,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.942505
4,DU-R-0018,NaN,DU-R-0018,2.0,13.09,4.0,SP19-026705,SP19-026705,0,0,...,13.193703,NaN,15.063655,NaN,15.063655,15.063655,15.063655,NaN,NaN,18.132786
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,DU-R-0189,NaN,DU-R-0189,1.0,11.31,3.0,NaN,NaN,0,0,...,11.455168,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
149,DU-R-0190,NaN,DU-R-0190,2.0,12.20,3.0,NaN,NaN,0,1,...,12.232717,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,DU-R-0191,NaN,DU-R-0191,2.0,13.77,3.0,NaN,NaN,0,0,...,13.864476,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
151,DU-R-0192,NaN,DU-R-0192,1.0,14.52,3.0,NaN,NaN,0,0,...,14.836413,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
%pip install tqdm
%pip install --upgrade ipywidgets
%pip install --upgrade jupyter

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


### Image Analysis

In [8]:
from enum import Enum
from dataclasses import dataclass
from pathlib import Path
import glob
import os
from tqdm.notebook import tqdm

HOVERNEXT_PREDICTIONS_DIR = DATA_DIR / "hover_next_predictions"
SLIDES_DIR = DATA_DIR / "digitized_slides"

class CellType(Enum):
    CONNECTIVE_TISSUE = "connective-tissue-cell"
    EOSINOPHIL = "eosinophil"
    EPITHELIAL_CELL = "epithelial-cell"
    LYMPHOCYTE = "lymphocyte"
    MITOSIS = "mitosis"
    NEUTROPHIL = "neutrophil"
    PLASMA_CELL = "plasma-cell"

@dataclass(eq=True, frozen=True)
class Cell:
    celltype : CellType
    x : float
    y : float

class PreAnnotatedSlide:
    def __init__(self, path_to_svs : Path, model_predictions_root : Path):
        if not path_to_svs.exists() or not model_predictions_root.exists(): raise ValueError("Please ensure both paths are valid and exist")
        self.path_to_svs = path_to_svs
        all_tsv_files_in_root = glob.glob("*.tsv", root_dir = model_predictions_root)
        if not all_tsv_files_in_root: raise ValueError(f"No TSV files found in {model_predictions_root}")
        
        all_cells_df = pd.read_csv(model_predictions_root / all_tsv_files_in_root[0], sep="\t")
        # TODO: refactor so concat isn't being called inside a loop
        # sue me...
        for tsv_file in all_tsv_files_in_root:
            this_cell_df = pd.read_csv(model_predictions_root / tsv_file, sep="\t")
            all_cells_df = pd.concat([all_cells_df, this_cell_df])

        self.cells = set()
        for _, row in all_cells_df.iterrows():
            cell = Cell(CellType(row["name"]), row["x"], row["y"])
            self.cells.add(cell)

# wrap in a list to eagerly evaluate
for slide_path in tqdm(list(SLIDES_DIR.iterdir())):
    corresponding_model_predictions_dir = HOVERNEXT_PREDICTIONS_DIR / slide_path.stem
    pre_annotated_slide = PreAnnotatedSlide(slide_path, corresponding_model_predictions_dir)
    print(len(pre_annotated_slide.cells))

  0%|          | 0/546 [00:00<?, ?it/s]

503684
605626
499723
753013
549461
378107
283873


/tmp/ipykernel_1445043/761528111.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_cells_df = pd.concat([all_cells_df, this_cell_df])


118114
120669
644853
277634
579798


/tmp/ipykernel_1445043/761528111.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_cells_df = pd.concat([all_cells_df, this_cell_df])


151137
317785
505475
521857
446239
252366


/tmp/ipykernel_1445043/761528111.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_cells_df = pd.concat([all_cells_df, this_cell_df])


274452
364829
602570
655738
95364
1236
179907
157238
125656


/tmp/ipykernel_1445043/761528111.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_cells_df = pd.concat([all_cells_df, this_cell_df])


286328


KeyboardInterrupt: 

In [ ]:
started_on_anti_tnf = df[(df["init_biologic_specified"] == 1) | (df["init_biologic_specified"] == 2)]
started_on_anti_tnf

In [ ]:
%pip install seaborn

In [ ]:
import seaborn as sns

df["days_from_office_vis_to_yr_fu"] = (df["age_at_yr_fu_date"] - df["age_at_date_officevis"]) * 365.25
sns.histplot(data=df, x="days_from_office_vis_to_yr_fu", kde=True, bins=20, kde_kws={"bw_method" : "silverman"})
plt.axvline(3 * 365.25, ls="--", color="gray", label="3 Years")
plt.axvline(5 * 365.25, ls="--", color="gray", label="5 Years")
plt.xlabel("Days Elapsed From 1st Office Visit to Longitudinal Follow-Up")
plt.legend()
plt.show()

In [ ]:
df["days_to_date_enrolled"] = round((df["age_at_date_enrolled"] - df["age_at_officevis"]) * 365.25)
df["days_to_radio"] = round((df["age_at_radio_date"] - df["age_at_officevis"]) * 365.25)


In [ ]:
from lifelines.utils import to_long_format
from lifelines.utils import add_covariate_to_timeline
import numpy as np

base_df = df[["study_id", "diagnosis", "age_calculated", "sex_v2"]]
base_df["duration"] = 10
base_df = to_long_format(base_df, duration_col="duration")
base_df

# time invariant covariates:
# diagnosis
# baseline age
# sex

# time varying covariates:
### Office Visits
# weight (age_at_weight_date, age_at_fu_date)
# height (age_at_height_date, age_at_fu_height_date)

### CBC
# hgb
# hct
# plateletcount
# wbc
# neutrophils
# lymphocytes
# eosinophils

### CMP
# esr
# crp
# albumin
# urea_calculated
# creatinine
# ast
# alt
# alp

### Stool Sample
# calprotectin